adjusted hw7

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [20]:
import torch
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms
import time
import torch.optim as optim
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import pandas as pd


In [21]:
# select device
if torch.cuda.is_available():
    device = torch.device("cuda")

elif torch.backends.mps.is_available():
    device = torch.device("mps")

else:
    device = torch.device("cpu")

print("Device:", device)

Device: mps


In [22]:
batch_size = 32

# Image transformations
train_transforms = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.8, 1.0)),
    #transforms.RandomHorizontalFlip(),
    #transforms.RandomVerticalFlip(),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

test_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

# Load dataset (without transforms)
full_dataset = datasets.ImageFolder(
    #root="/content/drive/MyDrive/gervigreing_lokaverk/filter",
    root="datageo/filter",
    transform=None
)

# Split train, val, test
train_data, val_data, test_data = random_split(
    full_dataset,
    [1120, 56, 224], # 80%, 4%, 16%
    generator=torch.Generator().manual_seed(42)
)

In [23]:
# only apply augmentations to training set
# create datasets
class CustomDataset(torch.utils.data.Dataset):
    def __init__(self, dataset, transform=None):
        self.dataset = dataset
        self.transform = transform

    def __getitem__(self, index):
        x, y = self.dataset[index]

        if self.transform:
            x = self.transform(x)

        return x, y

    def __len__(self):
        return len(self.dataset)


train_dataset = CustomDataset(train_data, transform=train_transforms)
val_dataset = CustomDataset(val_data, transform=test_transforms)
test_dataset = CustomDataset(test_data, transform=test_transforms)

# Create DataLoaders
if device.type == 'cuda': # running on colab
    train_loader = DataLoader(
        train_dataset, batch_size=batch_size, shuffle=True, pin_memory=True, num_workers=2
    )

    val_loader = DataLoader(
        val_dataset, batch_size=128, shuffle=False, pin_memory=True, num_workers=2
    )

    test_loader = DataLoader(
        test_dataset, batch_size=128, shuffle=False, pin_memory=True, num_workers=2
    )
    print('Running on Cuda')
elif device.type == 'mps': # running on mac
    train_loader = DataLoader(
        train_dataset, batch_size=batch_size, shuffle=True
    )
    
    val_loader = DataLoader(
        val_dataset, batch_size=128, shuffle=False
    )
    
    test_loader = DataLoader(
        test_dataset, batch_size=128, shuffle=False
    )
    print('Running on MPS')
else:
    print('error')

print(f"Training dataset size: {len(train_dataset)}")
print(f"Validation dataset size: {len(val_dataset)}")
print(f"Test dataset size: {len(test_dataset)}")

Running on MPS
Training dataset size: 1120
Validation dataset size: 56
Test dataset size: 224


In [24]:
# define resnet model
import torch.nn as nn
from torchvision import models

class ResNet18_Classifier(nn.Module):

    def __init__(self, hidden_size=256, dropout=0.3):
        super().__init__()

        self.model = models.resnet18(weights=None)

        input_size = self.model.fc.in_features
        print(input_size)

        # header 
        self.model.fc = nn.Sequential(
            nn.Linear(input_size, hidden_size),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_size, 2)
            )

    def forward(self, x):
        return self.model(x)

In [25]:
# parameter count 
def get_param_count(model):
    return sum(p.numel() for p in model.parameters())

# evaluation from hw7
def evaluate(model, loss_fun, dataloader, device):
    # Compute model predictions, loss and error rate
    model.eval()
    y_true, y_pred = [], []
    total, total_loss, incorrect = 0, 0, 0
    with torch.no_grad():
        for data, target in dataloader:
            data = data.to(device)
            target = target.to(device)

            output = model(data)

            total_loss += loss_fun(output, target).item() * target.size(0)

            _, pred = torch.max(output, 1)

            y_true.append(target.cpu())
            y_pred.append(pred.cpu())

            total += target.size(0)

            incorrect += pred.ne(target).sum().item()

    return total_loss / total, incorrect / total, y_true, y_pred

In [ ]:
# training function from hw7 adjusted
def train(model, optimizer, n_epochs, loss_history):
    for epoch in range(n_epochs):
        tstart = time.time()
        train_loss = 0
        model.train()
        for batch_idx, (data, target) in enumerate(train_loader):
            data = data.to(device)
            target = target.to(device)

            # Reset gradients
            optimizer.zero_grad()

            # Forward pass
            output = model(data)

            # Compute loss
            loss = F.cross_entropy(output, target)
            train_loss += loss.item() * target.size(0)

            # Backpropagation
            loss.backward()

            # Update weights
            optimizer.step()

        train_loss /= len(train_loader.dataset)

        # Evaluate on validation set
        val_loss, val_error, _, _ = evaluate(
            model, F.cross_entropy, val_loader, device
        )
        
        # export for monitor notebook
        loss_history['train_loss'].append(train_loss)
        loss_history['val_loss'].append(val_loss)
        loss_history['val_error'].append(val_error)
        loss_history['epoch'].append(epoch)

        # Save results after each epoch
        pd.DataFrame(loss_history).to_csv(
            "training_history.csv", index=False
        )

        print(
            f'Epoch: {epoch + 1:d}. '
            f'Training loss: {train_loss:.4f}, '
            f'validation loss: {val_loss:.4f}, '
            f'validation error: {val_error:.4f} '
            f'({time.time() - tstart:.2f} seconds)'
        )
    return model, loss_history

In [27]:
# plot training
def plot_training(loss_history):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))

    ax1.plot(loss_history['train_loss'], label='train')
    ax1.plot(loss_history['val_loss'], label='val')

    ax1.legend()
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Loss')
    ax1.set_title('Loss')

    ax2.plot(loss_history['val_error'], color='tab:orange')
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('Error rate')
    ax2.set_title('Error rate')

    plt.show()

In [36]:
# Run model
model = ResNet18_Classifier()

model.to(device)

print("Number of model parameters:", get_param_count(model))

# number of epochs
n_epochs = 50

# optimizers
#optimizer = optim.AdamW(model.parameters(), lr=0.0001)
optimizer = optim.SGD(model.parameters(), lr=0.001, momentum =0.9, weight_decay=1e-4)

loss_history = {
    'train_loss': [],
    'val_loss': [],
    'val_error': [],
    'epoch': []
}

model, loss_history = train(
    model, optimizer, n_epochs, loss_history
)
plot_training(loss_history)

# evaluate test set performance 
test_loss, test_error, _, _ = evaluate(
    model,
    F.cross_entropy,
    test_loader,
    device
)
print(f'Test set accuracy: {100*(1-test_error):.2f}%')

512
Number of model parameters: 11308354
Epoch: 1. Training loss: 0.4111, validation loss: 0.2169, validation error: 0.0357 (6.57 seconds)
Epoch: 2. Training loss: 0.1454, validation loss: 0.1385, validation error: 0.0357 (6.46 seconds)
Epoch: 3. Training loss: 0.0874, validation loss: 0.1175, validation error: 0.0357 (6.39 seconds)
Epoch: 4. Training loss: 0.0799, validation loss: 0.1751, validation error: 0.0714 (6.54 seconds)
Epoch: 5. Training loss: 0.0962, validation loss: 0.0979, validation error: 0.0536 (6.32 seconds)
Epoch: 6. Training loss: 0.0732, validation loss: 0.0990, validation error: 0.0536 (6.35 seconds)
Epoch: 7. Training loss: 0.0590, validation loss: 0.1459, validation error: 0.0536 (6.39 seconds)
Epoch: 8. Training loss: 0.0590, validation loss: 0.1465, validation error: 0.0357 (6.47 seconds)
Epoch: 9. Training loss: 0.0755, validation loss: 0.1587, validation error: 0.0893 (6.34 seconds)


KeyboardInterrupt: 

- with augmentations
    40 epochs
    optimizer = optim.SGD(model.parameters(), lr=0.001, momentum =0.9, weight_decay=1e-4)
    test acc 99.11%
    
- no augmentations
    40 epochs
    optimizer = optim.SGD(model.parameters(), lr=0.001, momentum =0.9, weight_decay=1e-4)
    test acc 99.11%

In [ ]:
# confusion matrix
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    accuracy_score
)

def plot_confusion_matrix(model, dataloader, class_names, device):

    # Get predictions using existing evaluation function
    test_loss, test_error, y_true, y_pred = evaluate(
        model, F.cross_entropy, dataloader, device
    )

    # Combine batches into single arrays
    y_true = torch.cat(y_true).numpy()
    y_pred = torch.cat(y_pred).numpy()

    # Calculate confusion matrix
    cm = confusion_matrix(
        y_true, y_pred,
        labels=np.arange(len(class_names))
    )

    # Normalize by actual class
    cm_normalized = cm.astype(float) / np.maximum(
        cm.sum(axis=1, keepdims=True), 1
    )

    # Plot
    plt.figure(figsize=(18, 15))

    sns.heatmap(
        cm_normalized,
        annot=False,
        cmap="Blues",
        xticklabels=class_names,
        yticklabels=class_names
    )

    plt.xlabel("Predicted class")
    plt.ylabel("Actual class")
    plt.title("ResNet-18 Confusion Matrix - RESISC45")

    plt.xticks(rotation=90)
    plt.yticks(rotation=0)

    plt.tight_layout()
    plt.show()

    # Print classification report
    print("\nClassification Report:")
    print(classification_report(
        y_true,
        y_pred,
        labels=np.arange(len(class_names)),
        target_names=class_names,
        zero_division=0
    ))

    print(f"Overall accuracy: {accuracy_score(y_true, y_pred)*100:.2f}%")
    
class_names = full_dataset.classes

plot_confusion_matrix(
    model,
    test_loader,
    class_names,
    device
)

In [ ]:
# accuracy for each class
def plot_class_accuracy(model, dataloader, class_names, device):

    # Evaluate model
    _, _, y_true, y_pred = evaluate(
        model, F.cross_entropy, dataloader, device
    )

    y_true = torch.cat(y_true).numpy()
    y_pred = torch.cat(y_pred).numpy()

    # Confusion matrix
    cm = confusion_matrix(
        y_true, y_pred,
        labels=np.arange(len(class_names))
    )

    # Calculate accuracy for each class
    class_accuracy = 100 * cm.diagonal() / np.maximum(
        cm.sum(axis=1), 1
    )

    # Sort classes by accuracy
    sorted_idx = np.argsort(class_accuracy)

    sorted_names = np.array(class_names)[sorted_idx]
    sorted_accuracy = class_accuracy[sorted_idx]

    # Plot
    plt.figure(figsize=(12, 12))
    plt.barh(sorted_names, sorted_accuracy)
    plt.xlabel("Accuracy (%)")
    plt.ylabel("Class")
    plt.title("ResNet-18 Accuracy per Class - RESISC45")
    plt.xlim(0, 100)
    plt.grid(axis="x", alpha=0.3)
    plt.tight_layout()
    plt.show()
    
plot_class_accuracy(
    model,
    test_loader,
    class_names,
    device
)